# 3.1 — SHAP sobre os modelos baseline

Calcula valores SHAP para as 15 combinações `(modelo, output)` da Etapa 2 e registra os resultados no experimento `reduzido` do MLflow.

**Fluxo (PLANO_3.md §3.1):**
1. Carregar dados de test e os 15 modelos baseline.
2. Selecionar explainer por arquitetura: `TreeExplainer` (DT/RF/XGBoost) ou `KernelExplainer` (SVR/ANN).
3. Calcular SHAP em `X_test` (D-E3-01) e logar:
   - matriz `.npy` em `ARTEFATOS/ETAPA_3/shap/`
   - métricas `shap_mean_<feature>` no MLflow
   - parâmetro `inputs_selecionados` (8 features — `CONVENCOES_MLFLOW.md`)
4. Gerar 15 bar plots e 15 beeswarm plots (todos os modelos × 3 outputs).
5. Consolidar tudo em `3.1_shap_consolidado.csv` (120 linhas).


## Seção 1 — Imports e configuração

In [1]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import shap

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_ROOT = Path("/Users/lorenzoferreira/Documents/UFRGS/TCC_SBO")
PROC_DIR     = PROJECT_ROOT / "ARTEFATOS" / "ETAPA_0" / "processed"
ETAPA2_ROOT  = PROJECT_ROOT / "ARTEFATOS" / "ETAPA_2"
SHAP_DIR     = PROJECT_ROOT / "ARTEFATOS" / "ETAPA_3" / "shap"
SHAP_DIR.mkdir(parents=True, exist_ok=True)

TRACKING_URI = f"file:///{PROJECT_ROOT}/mlruns"
EXPERIMENT   = "reduzido"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT)

print(f"shap   : {shap.__version__}")
print(f"mlflow : {mlflow.__version__}")
print(f"SHAP_DIR: {SHAP_DIR}")


shap   : 0.51.0
mlflow : 3.11.1
SHAP_DIR: /Users/lorenzoferreira/Documents/UFRGS/TCC_SBO/ARTEFATOS/ETAPA_3/shap


## Seção 2 — Carga de dados e modelos

- `X_train` e `X_test` no espaço normalizado (consistente com o que os modelos esperam).
- 8 features na ordem fixa: `P1, T1, T2, RRC1, BRC1, RRC2, BRC2, RFF`.
- Background do `KernelExplainer`: amostra de 100 pontos de `X_train` (`shap.sample` com `random_state=42`).


In [2]:
FEATURE_NAMES = ["P1", "T1", "T2", "RRC1", "BRC1", "RRC2", "BRC2", "RFF"]
MODELS  = ["SVR", "DT", "RF", "XGBoost", "ANN"]
OUTPUTS = ["ET", "M_CH3OH", "x_CH3OH"]

X_train = np.load(PROC_DIR / "X_train.npy")
X_test  = np.load(PROC_DIR / "X_test.npy")
assert X_train.shape[1] == 8 and X_test.shape[1] == 8

print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")

KERNEL_BACKGROUND = shap.sample(X_train, 100, random_state=RANDOM_STATE)
print(f"Background KernelExplainer: {KERNEL_BACKGROUND.shape}")


X_train: (1536, 8)
X_test : (193, 8)
Background KernelExplainer: (100, 8)


In [3]:
SKLEARN_MODELS    = {"SVR", "DT", "RF", "XGBoost"}
ANN_ARTIFACT_NAME = "model_v2.keras"  # versão final usada na comparação 2.3

def baseline_artifact_path(modelo: str, output: str) -> Path:
    if modelo == "ANN":
        return ETAPA2_ROOT / "ANN" / output / ANN_ARTIFACT_NAME
    return ETAPA2_ROOT / modelo / output / "model.pkl"

def load_baseline_model(modelo: str, output: str):
    path = baseline_artifact_path(modelo, output)
    if not path.exists():
        raise FileNotFoundError(path)
    if modelo == "ANN":
        from tensorflow import keras
        return keras.models.load_model(path)
    import pickle
    with open(path, "rb") as f:
        return pickle.load(f)

# Validação leve: 15 artefatos presentes
faltando = [(m, o) for m in MODELS for o in OUTPUTS
            if not baseline_artifact_path(m, o).exists()]
assert not faltando, f"Artefatos ausentes: {faltando}"
print("OK — 15/15 artefatos baseline acessíveis.")


OK — 15/15 artefatos baseline acessíveis.


## Seção 3 — Cálculo SHAP por (modelo, output)

Explainers (PLANO_3.md §3.1):

| Modelo | Explainer | Notas |
|--------|-----------|-------|
| DT, RF, XGBoost | `TreeExplainer` | Exato, segundos |
| SVR | `KernelExplainer` | Background K=100 |
| ANN | `KernelExplainer` | Background K=100 (fallback para `DeepExplainer` se >10 min) |

Por (modelo, output):
1. Carrega o modelo.
2. Calcula SHAP em `X_test` (D-E3-01).
3. Salva matriz em `ARTEFATOS/ETAPA_3/shap/<modelo>_<output>_shap.npy`.
4. Abre run MLflow no experimento `reduzido` com tags `phase=shap`, `model`, `output`; loga `shap_mean_<feature>`, parâmetro `inputs_selecionados` (8 originais) e a matriz SHAP como artefato.


In [4]:
def make_explainer(modelo: str, model_obj):
    """Retorna (explainer, shap_kwargs) apropriado para a arquitetura."""
    if modelo in ("DT", "RF", "XGBoost"):
        return shap.TreeExplainer(model_obj), {}
    if modelo == "SVR":
        # SVR.predict retorna shape (n,) — direto compatível com KernelExplainer
        return shap.KernelExplainer(model_obj.predict, KERNEL_BACKGROUND), {"nsamples": "auto"}
    if modelo == "ANN":
        # ANN.predict retorna (n, 1) — achatar para (n,)
        pred_fn = lambda X: model_obj.predict(X, verbose=0).flatten()
        return shap.KernelExplainer(pred_fn, KERNEL_BACKGROUND), {"nsamples": "auto"}
    raise ValueError(modelo)


def shap_values_2d(explainer, X, modelo, shap_kwargs):
    """Calcula SHAP e devolve matriz (n_samples, n_features) consistente."""
    sv = explainer.shap_values(X, **shap_kwargs) if shap_kwargs else explainer.shap_values(X)
    # TreeExplainer pode devolver lista (multi-output) ou array; aqui modelos são single-output
    if isinstance(sv, list):
        sv = sv[0]
    sv = np.asarray(sv)
    if sv.ndim == 3:
        # caso (n, n_features, n_outputs) com n_outputs=1
        sv = sv[..., 0]
    assert sv.shape == X.shape, f"SHAP shape {sv.shape} != X shape {X.shape}"
    return sv


In [5]:
shap_records = []  # para a tabela consolidada
shap_matrices = {}  # (modelo, output) -> matriz SHAP (n_test, 8)
timings = {}

for modelo in MODELS:
    for output in OUTPUTS:
        t0 = time.time()
        print(f"\n=== {modelo} × {output} ===")

        model_obj = load_baseline_model(modelo, output)
        explainer, kw = make_explainer(modelo, model_obj)
        sv = shap_values_2d(explainer, X_test, modelo, kw)

        # |SHAP| médio por feature
        shap_mean_abs = np.abs(sv).mean(axis=0)
        elapsed = time.time() - t0
        timings[(modelo, output)] = elapsed
        print(f"  shape={sv.shape}  tempo={elapsed:5.1f}s")

        # Salvar matriz no disco
        npy_path = SHAP_DIR / f"{modelo}_{output}_shap.npy"
        np.save(npy_path, sv)
        shap_matrices[(modelo, output)] = sv

        # Registrar no MLflow
        with mlflow.start_run(run_name=f"shap_{modelo}_{output}"):
            mlflow.set_tag("phase", "shap")
            mlflow.set_tag("model", modelo)
            mlflow.set_tag("output", output)

            mlflow.log_param("random_state", RANDOM_STATE)
            mlflow.log_param("explainer", type(explainer).__name__)
            mlflow.log_param("background_k", len(KERNEL_BACKGROUND) if modelo in ("SVR", "ANN") else "n/a")
            mlflow.log_param("n_eval", int(X_test.shape[0]))
            mlflow.log_param("eval_split", "X_test")
            mlflow.log_param("inputs_selecionados", str(FEATURE_NAMES))
            mlflow.log_param("k", 8)

            for fname, val in zip(FEATURE_NAMES, shap_mean_abs):
                mlflow.log_metric(f"shap_mean_{fname}", float(val))
            mlflow.log_metric("runtime_seconds", float(elapsed))

            mlflow.log_artifact(str(npy_path), artifact_path="shap")

        # Acumular registros para a tabela consolidada
        ranks = (-shap_mean_abs).argsort().argsort() + 1
        for f_idx, fname in enumerate(FEATURE_NAMES):
            shap_records.append({
                "modelo":    modelo,
                "output":    output,
                "feature":   fname,
                "shap_mean": float(shap_mean_abs[f_idx]),
                "shap_rank": int(ranks[f_idx]),
            })

print("\nSHAP concluído para as 15 combinações.")



=== SVR × ET ===


  0%|          | 0/193 [00:00<?, ?it/s]

  shape=(193, 8)  tempo=149.4s

=== SVR × M_CH3OH ===


  0%|          | 0/193 [00:00<?, ?it/s]

  shape=(193, 8)  tempo=151.4s

=== SVR × x_CH3OH ===


...

  shape=(193, 8)  tempo= 42.3s

=== ANN × M_CH3OH ===


  0%|          | 0/193 [00:00<?, ?it/s]

  shape=(193, 8)  tempo= 39.4s

=== ANN × x_CH3OH ===


  0%|          | 0/193 [00:00<?, ?it/s]

  shape=(193, 8)  tempo= 39.8s

SHAP concluído para as 15 combinações.


## Seção 4 — Visualizações por (modelo, output)

In [6]:
def bar_plot_shap(shap_mean_abs, feature_names, title, out_path):
    order = np.argsort(shap_mean_abs)[::-1]
    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.barh(
        [feature_names[i] for i in order][::-1],
        shap_mean_abs[order][::-1],
        color="#3b7dd8",
    )
    ax.set_xlabel("|SHAP| médio")
    ax.set_title(title)
    ax.grid(True, axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


for (modelo, output), sv in shap_matrices.items():
    shap_mean_abs = np.abs(sv).mean(axis=0)
    out_path = SHAP_DIR / f"3.1_bar_{modelo}_{output}.png"
    bar_plot_shap(shap_mean_abs, FEATURE_NAMES, f"|SHAP| médio — {modelo} × {output}", out_path)

print(f"15 bar plots salvos em {SHAP_DIR}")


15 bar plots salvos em /Users/lorenzoferreira/Documents/UFRGS/TCC_SBO/ARTEFATOS/ETAPA_3/shap


In [7]:
# Beeswarm para todos os 5 modelos × 3 outputs
# Carrega diretamente do disco para não depender de shap_matrices em memória
ALL_MODELS = ["SVR", "DT", "RF", "XGBoost", "ANN"]

for modelo in ALL_MODELS:
    for output in OUTPUTS:
        sv = np.load(SHAP_DIR / f"{modelo}_{output}_shap.npy")
        explanation = shap.Explanation(
            values=sv,
            data=X_test,
            feature_names=FEATURE_NAMES,
        )
        plt.figure()
        shap.plots.beeswarm(explanation, show=False, max_display=8)
        fig = plt.gcf()
        fig.suptitle(f"Beeswarm — {modelo} × {output}", fontsize=11)
        out_path = SHAP_DIR / f"3.1_beeswarm_{modelo}_{output}.png"
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

print(f"{len(ALL_MODELS) * len(OUTPUTS)} beeswarm plots salvos em {SHAP_DIR}")


15 beeswarm plots salvos em /Users/lorenzoferreira/Documents/UFRGS/TCC_SBO/ARTEFATOS/ETAPA_3/shap


## Seção 5 — Tabela consolidada

`3.1_shap_consolidado.csv` — 15 modelos × 8 features = 120 linhas.


In [ ]:
shap_consolidado = pd.DataFrame(shap_records)
assert len(shap_consolidado) == 15 * 8, f"Esperado 120 linhas, obtido {len(shap_consolidado)}"
assert shap_consolidado.notna().all().all(), "Existe NaN no consolidado de SHAP."

csv_path = SHAP_DIR / "3.1_shap_consolidado.csv"
shap_consolidado.to_csv(csv_path, index=False)
print(f"Salvo: {csv_path}  ({len(shap_consolidado)} linhas)")
shap_consolidado.head(10) # Esperado 120 linhas


Salvo: /Users/lorenzoferreira/Documents/UFRGS/TCC_SBO/ARTEFATOS/ETAPA_3/shap/3.1_shap_consolidado.csv  (120 linhas)


,modelo,output,feature,shap_mean,shap_rank
0,SVR,ET,P1,0.004348,7
1,SVR,ET,T1,0.028899,4
2,SVR,ET,T2,0.002549,8
3,SVR,ET,RRC1,0.008706,6
4,SVR,ET,BRC1,0.079866,2
5,SVR,ET,RRC2,0.040686,3
6,SVR,ET,BRC2,0.009378,5
7,SVR,ET,RFF,0.094018,1
8,SVR,M_CH3OH,P1,0.011172,7
9,SVR,M_CH3OH,T1,0.082824,2


In [9]:
# Visão rápida: |SHAP| médio por (modelo, feature) agregado nos 3 outputs
pivot_modelo_feature = (
    shap_consolidado
    .groupby(["modelo", "feature"], as_index=False)["shap_mean"].mean()
    .pivot(index="modelo", columns="feature", values="shap_mean")
    [FEATURE_NAMES]
    .round(4)
)
print("|SHAP| médio (média sobre os 3 outputs):")
pivot_modelo_feature


|SHAP| médio (média sobre os 3 outputs):


feature,P1,T1,T2,RRC1,BRC1,RRC2,BRC2,RFF
modelo,,,,,,,,
ANN,0.0052,0.0425,0.0011,0.0698,0.0777,0.0395,0.0292,0.0641
DT,0.0043,0.0349,0.0015,0.0695,0.0750,0.0320,0.0206,0.0616
RF,0.0029,0.0348,0.0007,0.0670,0.0748,0.0288,0.0182,0.0601
SVR,0.0071,0.0422,0.0036,0.0674,0.0741,0.0377,0.0284,0.0622
XGBoost,0.0048,0.0382,0.0018,0.0702,0.0767,0.0359,0.0235,0.0623


In [10]:
# Resumo de tempo por modelo (útil para D-E3-02: fallback DeepExplainer se ANN > 10 min/output)
timing_df = (
    pd.DataFrame(
        [(m, o, t) for (m, o), t in timings.items()],
        columns=["modelo", "output", "segundos"],
    )
    .pivot(index="modelo", columns="output", values="segundos")
    .round(1)
)
print("Tempo (s) por (modelo, output):")
timing_df


Tempo (s) por (modelo, output):


output,ET,M_CH3OH,x_CH3OH
modelo,,,
ANN,42.3,39.4,39.8
DT,0.0,0.0,0.0
RF,2.6,1.5,3.0
SVR,149.4,151.4,118.8
XGBoost,0.1,0.0,0.0


## Validação final

- 15 matrizes `.npy` em `ARTEFATOS/ETAPA_3/shap/`.
- 15 bar plots + 15 beeswarms (todos os modelos × 3 outputs).
- `3.1_shap_consolidado.csv` com 120 linhas, sem NaN.
- 15 runs `phase=shap` registradas no experimento `reduzido` (logando `shap_mean_<feature>`, `inputs_selecionados`).


In [11]:
# Validação fechada
n_npy      = len(list(SHAP_DIR.glob("*_shap.npy")))
n_bars     = len(list(SHAP_DIR.glob("3.1_bar_*.png")))
n_beeswarm = len(list(SHAP_DIR.glob("3.1_beeswarm_*.png")))

runs_reduzido = mlflow.search_runs(experiment_names=[EXPERIMENT])
shap_runs = runs_reduzido[runs_reduzido.get("tags.phase", "") == "shap"]

print(f"matrizes SHAP (.npy)  : {n_npy}  (esperado 15)")
print(f"bar plots             : {n_bars}  (esperado 15)")
print(f"beeswarm plots        : {n_beeswarm}  (esperado 15)")
print(f"runs MLflow (phase=shap): {len(shap_runs)}  (esperado 15)")
print(f"consolidado linhas    : {len(shap_consolidado)}  (esperado 120)")

assert n_npy == 15
assert n_bars == 15
assert n_beeswarm == 15
assert len(shap_consolidado) == 120
print("\nOK — Etapa 3.1 validada.")


matrizes SHAP (.npy)  : 15  (esperado 15)
bar plots             : 15  (esperado 15)
beeswarm plots        : 15  (esperado 15)
runs MLflow (phase=shap): 15  (esperado 15)
consolidado linhas    : 120  (esperado 120)

OK — Etapa 3.1 validada.
